# Fixed Data Preparation Pipeline — split OCR/LLM + timing

This notebook keeps the same `rawData → ocr_texts → output` path style, separates OCR and LLM execution into different stages, and writes per-request timing reports.


## Setup + configuration

In [ ]:
import os, re, json, csv, shutil, time, tempfile, threading
from pathlib import Path
from collections import deque
from pdf2image import convert_from_path
from dotenv import load_dotenv
from PIL import Image
import requests as req
from typhoon_ocr import ocr_document

load_dotenv()

TYPHOON_KEY = os.getenv("TYPHOON_KEY", "")
PROVINCE = "อุบลราชธานี"
CONSTITUENCY = 2
RAW_DATA_DIR = "rawData"
OUTPUT_DIR = "output"
OCR_TEXT_DIR = "ocr_texts"

Path(OUTPUT_DIR).mkdir(exist_ok=True)
Path(OCR_TEXT_DIR).mkdir(exist_ok=True)

OCR_BASE_URL = "https://api.opentyphoon.ai/v1"
LLM_BASE_URL = "https://api.opentyphoon.ai/v1"
OCR_MODEL = "typhoon-ocr"
LLM_MODEL = "typhoon-v2.5-30b-a3b-instruct"

OCR_LIMIT_PER_MIN = 20
LLM_LIMIT_PER_MIN = 200
OCR_PAGE_TIMEOUT = 60
MAX_IMAGE_SIDE = 2200
MAX_IMAGE_BYTES = 4_000_000

os.environ["TYPHOON_OCR_API_KEY"] = TYPHOON_KEY
os.environ["OPENAI_API_KEY"] = TYPHOON_KEY

VALID_PARTIES_TEXT = '''
ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เพื่อไทย, ภูมิใจไทย,
สังคมประชาธิปไตยไทย, รักชาติ, ประชาธิปไตยใหม่, ครูไทยเพื่อประชาชน,
ประชาชน, ไทยก้าวใหม่, เสรีรวมไทย, พลังไทยรักชาติ, เพื่อชีวิตใหม่,
ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม,
ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, ประชาชาติ, แผ่นดินธรรม, คลองไทย,
พลังประชารัฐ, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, กรีน, วิชชั่นใหม่,
พลวัต, กล้าธรรม, ไทยรวมไทย, ฟิวชัน, พลังสังคมใหม่, ไทยสร้างไทย,
รวมไทยสร้างชาติ, มิติใหม่, ไทยภักดี, ไทยพิทักษ์ธรรม, ไทยชนะ,
ไทรวมพลัง, ก้าวอิสระ, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, ไทยก้าวหน้า,
พร้อม, รวมใจไทย, ประชาอาสาชาติ, ไทยทรัพย์ทวี, รวมพลังประชาชน,
เพื่อบ้านเมือง, อนาคตไทย, เครือข่ายชาวนาแห่งประเทศไทย
'''


# Runtime timing logs. These are reset when this setup cell is re-run.
OCR_REQUEST_TIMINGS = []
OCR_FILE_TIMINGS = []
LLM_REQUEST_TIMINGS = []
LLM_FILE_TIMINGS = []


## Rate limiter

In [2]:
class RateLimiter:
    def __init__(self, max_calls, period=60):
        self.max_calls = max_calls
        self.period = period
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.time()
            while self.calls and now - self.calls[0] >= self.period:
                self.calls.popleft()
            if len(self.calls) >= self.max_calls:
                sleep_time = self.period - (now - self.calls[0]) + 0.05
                time.sleep(max(0, sleep_time))
                now = time.time()
                while self.calls and now - self.calls[0] >= self.period:
                    self.calls.popleft()
            self.calls.append(time.time())

ocr_limiter = RateLimiter(OCR_LIMIT_PER_MIN, 60)
llm_limiter = RateLimiter(LLM_LIMIT_PER_MIN, 60)

def format_seconds(seconds):
    return f"{seconds:.2f}s"

def write_timing_reports():
    Path(OUTPUT_DIR).mkdir(exist_ok=True)

    def _write_csv(path, rows):
        if not rows:
            return
        fieldnames = list(rows[0].keys())
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"Timing report → {path} ({len(rows)} rows)")

    _write_csv(Path(OUTPUT_DIR) / "ocr_request_timings.csv", OCR_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "ocr_file_timings.csv", OCR_FILE_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "llm_request_timings.csv", LLM_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "llm_file_timings.csv", LLM_FILE_TIMINGS)


## Helpers: parsing paths + numbers

In [3]:
def thai_to_int(value):
    if value is None:
        return None
    thai_digits = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    s = str(value).translate(thai_digits).strip()
    s = s.replace(",", "").replace(" ", "")
    m = re.search(r"-?\d+", s)
    return int(m.group(0)) if m else None

def detect_form_type(pdf_path):
    return "party_list" if "บช" in Path(pdf_path).name else "constituency"

def extract_path_metadata(pdf_path):
    pdf = Path(pdf_path)
    parts = pdf.parts
    try:
        amphoe = parts[-4]
        tambon = parts[-3]
        m = re.search(r"(\d+)$", parts[-2])
        unit_num = int(m.group(1)) if m else 0
    except Exception:
        amphoe, tambon, unit_num = "unknown", "unknown", 0
    return amphoe, tambon, unit_num

def llm_json_path(pdf_path):
    pdf = Path(pdf_path)
    raw_dir = Path(RAW_DATA_DIR)
    try:
        rel = pdf.relative_to(raw_dir)
    except ValueError:
        rel = Path(pdf.name)
    out = Path(OUTPUT_DIR) / rel.with_suffix(".json")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

def ocr_txt_path(pdf_path):
    pdf = Path(pdf_path)
    raw_dir = Path(RAW_DATA_DIR)
    try:
        rel = pdf.relative_to(raw_dir)
    except ValueError:
        rel = Path(pdf.name)
    out = Path(OCR_TEXT_DIR) / rel.parent / (pdf.stem + "_ocr.txt")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

## OCR helpers

In [4]:
def pdf_to_images(pdf_path, dpi=300):
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
            tmp = f.name
        shutil.copy2(pdf_path, tmp)
        return convert_from_path(tmp, dpi=dpi)
    finally:
        if tmp and os.path.exists(tmp):
            os.unlink(tmp)

def save_image_tmp(page_img, index):
    tmp_dir = Path("_tmp_imgs")
    tmp_dir.mkdir(exist_ok=True)
    img = page_img.convert("RGB")
    if max(img.size) > MAX_IMAGE_SIDE:
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    path = tmp_dir / f"page_{index:03d}.jpg"
    quality = 90
    while True:
        img.save(str(path), "JPEG", quality=quality, optimize=True)
        if path.stat().st_size <= MAX_IMAGE_BYTES or quality <= 55:
            break
        quality -= 10
    return str(path)

def ocr_pdf_to_text(pdf_path, force=False):
    txt_path = ocr_txt_path(pdf_path)
    if txt_path.exists() and txt_path.stat().st_size > 50 and not force:
        print(f"    ↩️  OCR cached: {txt_path.name}")
        return txt_path.read_text(encoding="utf-8")

    convert_start = time.perf_counter()
    pages = pdf_to_images(pdf_path)
    convert_seconds = time.perf_counter() - convert_start
    print(f"    🖼️  PDF→image: {format_seconds(convert_seconds)} ({len(pages)} pages)")

    all_text = []

    for i, page in enumerate(pages):
        img_path = save_image_tmp(page, i)
        img_bytes = Path(img_path).stat().st_size
        print(f"    🔍 OCR page {i+1}/{len(pages)} ({img_bytes/1_000_000:.2f} MB)...", end=" ", flush=True)
        page_result = {}
        page_error = {}

        def _ocr(_img=img_path):
            request_start = time.perf_counter()
            wait_start = time.perf_counter()
            try:
                ocr_limiter.wait()
                wait_seconds = time.perf_counter() - wait_start
                api_start = time.perf_counter()
                md = ocr_document(_img, model=OCR_MODEL, task_type="v1.5")
                api_seconds = time.perf_counter() - api_start
                total_seconds = time.perf_counter() - request_start
                page_result["text"] = md
                page_result["timing"] = {
                    "file": str(pdf_path),
                    "page": i + 1,
                    "status": "PASS",
                    "chars": len(md),
                    "image_mb": round(img_bytes / 1_000_000, 3),
                    "wait_seconds": round(wait_seconds, 3),
                    "api_seconds": round(api_seconds, 3),
                    "total_seconds": round(total_seconds, 3),
                }
            except Exception as e:
                total_seconds = time.perf_counter() - request_start
                page_error["err"] = str(e)
                page_result["timing"] = {
                    "file": str(pdf_path),
                    "page": i + 1,
                    "status": "FAIL",
                    "chars": 0,
                    "image_mb": round(img_bytes / 1_000_000, 3),
                    "wait_seconds": None,
                    "api_seconds": None,
                    "total_seconds": round(total_seconds, 3),
                    "error": str(e)[:200],
                }

        pt = threading.Thread(target=_ocr, daemon=True)
        pt.start()
        pt.join(timeout=OCR_PAGE_TIMEOUT)

        timing = page_result.get("timing")
        if pt.is_alive():
            timing = {
                "file": str(pdf_path),
                "page": i + 1,
                "status": "TIMEOUT",
                "chars": 0,
                "image_mb": round(img_bytes / 1_000_000, 3),
                "wait_seconds": None,
                "api_seconds": None,
                "total_seconds": OCR_PAGE_TIMEOUT,
                "error": f"timeout after {OCR_PAGE_TIMEOUT}s",
            }
            OCR_REQUEST_TIMINGS.append(timing)
            print(f"⏱️ page timeout ({OCR_PAGE_TIMEOUT}s) → skipped")
            all_text.append("")
            continue

        if timing:
            OCR_REQUEST_TIMINGS.append(timing)

        if "err" in page_error:
            err = page_error["err"]
            print(f"❌ {err[:100]} | total={format_seconds(timing['total_seconds']) if timing else 'n/a'}")
            all_text.append("")
            continue

        md = page_result.get("text", "")
        all_text.append(md)
        print(
            f"✓ chars={len(md)} "
            f"wait={format_seconds(timing['wait_seconds'])} "
            f"api={format_seconds(timing['api_seconds'])} "
            f"total={format_seconds(timing['total_seconds'])}"
        )

    full_text = "\n\n--- PAGE BREAK ---\n\n".join(all_text)
    if full_text.strip():
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"    💾 Saved → {txt_path}")
    else:
        print("    ⚠️ All pages empty — not cached")
    return full_text


## Run OCR stage

In [5]:
def run_ocr_stage(pdf_paths, force=False):
    ocr_failed = []
    stage_start = time.perf_counter()
    total = len(pdf_paths)

    for i, pdf in enumerate(pdf_paths):
        file_start = time.perf_counter()
        txt = ocr_txt_path(str(pdf))
        if txt.exists() and txt.stat().st_size > 50 and not force:
            elapsed = time.perf_counter() - file_start
            print(f"[{i+1}/{total}] OCR SKIP: {pdf.name} | total={format_seconds(elapsed)}")
            OCR_FILE_TIMINGS.append({"file": str(pdf), "status": "SKIP", "pages": None, "chars": txt.stat().st_size, "total_seconds": round(elapsed, 3)})
            continue

        print(f"[{i+1}/{total}] OCR: {pdf}")
        try:
            text = ocr_pdf_to_text(str(pdf), force=force)
            if not text.strip():
                raise ValueError("Empty OCR output")
            elapsed = time.perf_counter() - file_start
            page_count = text.count("--- PAGE BREAK ---") + 1 if text.strip() else 0
            OCR_FILE_TIMINGS.append({"file": str(pdf), "status": "PASS", "pages": page_count, "chars": len(text), "total_seconds": round(elapsed, 3)})
            print(f"  ✓ OCR FILE DONE: pages={page_count} chars={len(text)} total={format_seconds(elapsed)}")
        except Exception as e:
            elapsed = time.perf_counter() - file_start
            print(f"  ✗ OCR FAILED: {e} | total={format_seconds(elapsed)}")
            ocr_failed.append({"file": str(pdf), "error": str(e)})
            OCR_FILE_TIMINGS.append({"file": str(pdf), "status": "FAIL", "pages": None, "chars": 0, "total_seconds": round(elapsed, 3), "error": str(e)[:200]})

    if ocr_failed:
        path = Path(OUTPUT_DIR) / "_ocr_failed.json"
        path.write_text(json.dumps(ocr_failed, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"OCR failed: {len(ocr_failed)} → {path}")

    stage_elapsed = time.perf_counter() - stage_start
    print(f"OCR STAGE DONE: files={total}, failed={len(ocr_failed)}, total={format_seconds(stage_elapsed)}")
    write_timing_reports()
    return ocr_failed


## LLM prompt helpers

In [6]:
def compact_ocr_text(ocr_text, form_type):
    pages = re.split(r"\n\s*--- PAGE BREAK ---\s*\n", ocr_text)
    kept = []
    for page in pages:
        if "ลงชื่อ" in page and "<table" not in page and "จำนวนบัตรเลือกตั้ง" not in page:
            continue
        if any(k in page for k in ["จำนวนผู้มีสิทธิ", "จำนวนบัตรเลือกตั้ง", "<table", "รวมคะแนนทั้งสิ้น"]):
            kept.append(page)
    text = "\n\n--- PAGE BREAK ---\n\n".join(kept) if kept else ocr_text
    return text[:9000]

def build_llm_prompt(ocr_text, metadata, validation_feedback=None, previous_json=None):
    form_type = metadata.get("form_type") or detect_form_type(metadata.get("source", ""))
    compact_text = compact_ocr_text(ocr_text, form_type)
    result_label = "candidate rows from the constituency table; include candidate name if visible" if form_type == "constituency" else "party-list rows from all party-list pages"
    feedback = ""
    if validation_feedback:
        feedback = f"""
Previous extraction failed validation:
{json.dumps(validation_feedback, ensure_ascii=False)}
Fix the extracted values using the OCR text. Prefer numeric digits over Thai words if they conflict.
"""
    prev = ""
    if previous_json:
        prev = f"""
Previous JSON:
{json.dumps(previous_json, ensure_ascii=False)}
"""

    return f"""You are a Thai election data extraction system.
Return ONLY valid JSON. Do not include markdown. Do not explain.

Form type hint: {form_type}
Valid party names for correction:
{VALID_PARTIES}

Required JSON structure:
{{
  "form_type": "{form_type}",
  "summary": {{
    "eligible_voters": int_or_null,
    "turnout": int_or_null,
    "ballots_allocated": int_or_null,
    "ballots_used": int_or_null,
    "valid_ballots": int_or_null,
    "spoiled_ballots": int_or_null,
    "abstain_ballots": int_or_null,
    "ballots_remaining": int_or_null
  }},
  "results": [
    {{
      "number": int,
      "name": string_or_empty,
      "party": string,
      "votes": int,
      "votes_th": string_or_empty
    }}
  ]
}}

Rules:
- Extract {result_label}.
- Convert Thai digits to Arabic integers.
- Use null only if a summary value is truly missing.
- Use 0 for actual zero votes.
- Prefer numeric digits over Thai words when OCR conflicts.
- For constituency, party is the candidate's political party.
- For party_list, party is the party name and name should be empty.
- Preserve metadata is not required; it will be added by code.
{feedback}
{prev}
OCR TEXT:
{compact_text}
"""

## LLM API + JSON loading

In [7]:
def load_json_from_llm_text(raw):
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw).strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        raw = raw[start:end+1]
    return json.loads(raw)

def call_llm(prompt, request_label="llm", source=None):
    headers = {"Authorization": f"Bearer {TYPHOON_KEY}", "Content-Type": "application/json"}
    body = {"model": LLM_MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": 4096, "temperature": 0.0}

    for attempt in range(3):
        request_start = time.perf_counter()
        wait_start = time.perf_counter()
        llm_limiter.wait()
        wait_seconds = time.perf_counter() - wait_start
        api_start = time.perf_counter()

        try:
            resp = req.post(f"{LLM_BASE_URL}/chat/completions", headers=headers, json=body, timeout=90)
            api_seconds = time.perf_counter() - api_start
            total_seconds = time.perf_counter() - request_start

            timing_row = {
                "file": source,
                "request_label": request_label,
                "attempt": attempt + 1,
                "status_code": resp.status_code,
                "prompt_chars": len(prompt),
                "wait_seconds": round(wait_seconds, 3),
                "api_seconds": round(api_seconds, 3),
                "total_seconds": round(total_seconds, 3),
            }

            if resp.status_code == 429:
                timing_row["status"] = "RATE_LIMIT"
                LLM_REQUEST_TIMINGS.append(timing_row)
                print(
                    f"    ⏳ LLM {request_label} attempt={attempt+1} rate-limited "
                    f"wait={format_seconds(wait_seconds)} api={format_seconds(api_seconds)} total={format_seconds(total_seconds)}"
                )
                time.sleep(20 * (attempt + 1))
                continue

            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            timing_row["status"] = "PASS"
            timing_row["response_chars"] = len(content)
            LLM_REQUEST_TIMINGS.append(timing_row)
            print(
                f"    🤖 LLM {request_label} attempt={attempt+1} "
                f"prompt={len(prompt)} chars response={len(content)} chars "
                f"wait={format_seconds(wait_seconds)} api={format_seconds(api_seconds)} total={format_seconds(total_seconds)}"
            )
            return content

        except Exception as e:
            api_seconds = time.perf_counter() - api_start
            total_seconds = time.perf_counter() - request_start
            LLM_REQUEST_TIMINGS.append({
                "file": source,
                "request_label": request_label,
                "attempt": attempt + 1,
                "status": "FAIL",
                "status_code": getattr(locals().get("resp", None), "status_code", None),
                "prompt_chars": len(prompt),
                "wait_seconds": round(wait_seconds, 3),
                "api_seconds": round(api_seconds, 3),
                "total_seconds": round(total_seconds, 3),
                "error": str(e)[:200],
            })
            if attempt == 2:
                raise
            time.sleep(5 * (attempt + 1))

    raise RuntimeError("LLM rate limit retry exhausted")


## Normalization + validation

In [8]:
def normalize_record(data, metadata):
    form_type = data.get("form_type") or metadata.get("form_type") or detect_form_type(metadata.get("source", ""))
    data["form_type"] = form_type if form_type in ["constituency", "party_list"] else metadata.get("form_type")
    summary = data.get("summary") or {}
    for k in ["eligible_voters", "turnout", "ballots_allocated", "ballots_used", "valid_ballots", "spoiled_ballots", "abstain_ballots", "ballots_remaining"]:
        summary[k] = thai_to_int(summary.get(k))
    data["summary"] = summary

    clean_results = []
    for r in data.get("results", []):
        number = thai_to_int(r.get("number"))
        votes = thai_to_int(r.get("votes"))
        if number is None:
            continue
        clean_results.append({"number": number, "name": str(r.get("name", "") or "").strip(), "party": str(r.get("party", "") or "").strip(), "votes": votes if votes is not None else 0, "votes_th": str(r.get("votes_th", "") or "").strip()})
    data["results"] = sorted(clean_results, key=lambda x: x.get("number", 0))
    return data

def validate_record(data):
    errors = []
    warnings = []
    summary = data.get("summary", {})
    results = data.get("results", [])
    form_type = data.get("form_type")

    if form_type not in ["constituency", "party_list"]:
        errors.append("invalid form_type")
    if not results:
        errors.append("empty results")

    nums = [r.get("number") for r in results]
    if len(nums) != len(set(nums)):
        errors.append("duplicate result number")

    for r in results:
        if r.get("number") is None:
            errors.append("missing result number")
        if r.get("votes") is None:
            errors.append(f"missing votes for number {r.get('number')}")
        elif r.get("votes") < 0:
            errors.append(f"negative votes for number {r.get('number')}")
        if not r.get("party"):
            warnings.append(f"missing party for number {r.get('number')}")

    vote_sum = sum(r.get("votes", 0) for r in results)
    valid = summary.get("valid_ballots")
    used = summary.get("ballots_used")
    spoiled = summary.get("spoiled_ballots")
    abstain = summary.get("abstain_ballots")
    allocated = summary.get("ballots_allocated")
    remaining = summary.get("ballots_remaining")
    turnout = summary.get("turnout")

    if valid is not None and vote_sum != valid:
        errors.append(f"vote sum mismatch: results={vote_sum}, valid_ballots={valid}")
    if None not in [valid, spoiled, abstain, used] and valid + spoiled + abstain != used:
        errors.append(f"ballot used mismatch: valid+spoiled+abstain={valid + spoiled + abstain}, ballots_used={used}")
    if None not in [used, allocated] and used > allocated:
        errors.append(f"ballots_used greater than ballots_allocated: {used}>{allocated}")
    if None not in [used, remaining, allocated] and used + remaining != allocated:
        warnings.append(f"allocated mismatch: used+remaining={used + remaining}, allocated={allocated}")
    if None not in [turnout, used] and turnout != used:
        warnings.append(f"turnout and ballots_used differ: turnout={turnout}, ballots_used={used}")

    status = "REVIEW" if errors else "WARNING" if warnings else "PASS"
    return {"total_votes_in_table": vote_sum, "status": status, "errors": errors, "warnings": warnings}

## LLM parse + retry

In [9]:
def llm_parse_to_json(ocr_text, metadata, validation_feedback=None, previous_json=None):
    form_type = metadata.get("form_type") or detect_form_type(metadata.get("source", ""))
    metadata = dict(metadata)
    metadata["form_type"] = form_type
    prompt = build_llm_prompt(ocr_text, metadata, validation_feedback, previous_json)
    request_label = "retry" if validation_feedback else "initial"
    raw = call_llm(prompt, request_label=request_label, source=metadata.get("source"))
    data = load_json_from_llm_text(raw)
    data = normalize_record(data, metadata)
    validation = validate_record(data)
    return {"metadata": metadata, "form_type": data.get("form_type", form_type), "summary": data.get("summary", {}), "results": data.get("results", []), "_validation": validation}

def llm_parse_with_retry(ocr_text, metadata, max_retries=1):
    result = llm_parse_to_json(ocr_text, metadata)
    retry_count = 0
    for _ in range(max_retries):
        if result["_validation"]["status"] in ["PASS", "WARNING"]:
            break
        retry_count += 1
        print(f"    🔁 Validation retry {retry_count}: {result['_validation'].get('errors', [])}")
        result = llm_parse_to_json(
            ocr_text,
            metadata,
            validation_feedback=result["_validation"],
            previous_json={"form_type": result.get("form_type"), "summary": result.get("summary"), "results": result.get("results")},
        )
    result["_validation"]["retry_count"] = retry_count
    return result


## CSV/report helpers

In [10]:
def result_rows_from_json(result, pdf_path):
    amphoe, tambon, unit_num = extract_path_metadata(pdf_path)
    rows = []
    summary = result.get("summary", {})
    validation = result.get("_validation", {})
    for r in result.get("results", []):
        rows.append({"province": PROVINCE, "constituency": CONSTITUENCY, "amphoe": amphoe, "tambon": tambon, "unit": unit_num, "form_type": result.get("form_type", detect_form_type(pdf_path)), "eligible_voters": summary.get("eligible_voters"), "turnout": summary.get("turnout"), "ballots_allocated": summary.get("ballots_allocated"), "ballots_used": summary.get("ballots_used"), "valid_ballots": summary.get("valid_ballots"), "spoiled_ballots": summary.get("spoiled_ballots"), "abstain_ballots": summary.get("abstain_ballots"), "ballots_remaining": summary.get("ballots_remaining"), "candidate_number": r.get("number"), "candidate_name": r.get("name", ""), "party": r.get("party", ""), "votes": r.get("votes", 0), "votes_th": r.get("votes_th", ""), "validation": validation.get("status", ""), "validation_errors": "; ".join(validation.get("errors", [])), "validation_warnings": "; ".join(validation.get("warnings", [])), "source": str(pdf_path)})
    return rows

def validation_report_row(result, pdf_path):
    amphoe, tambon, unit_num = extract_path_metadata(pdf_path)
    v = result.get("_validation", {})
    return {"source": str(pdf_path), "form_type": result.get("form_type", detect_form_type(pdf_path)), "amphoe": amphoe, "tambon": tambon, "unit": unit_num, "status": v.get("status", ""), "total_votes_in_table": v.get("total_votes_in_table", 0), "valid_ballots": result.get("summary", {}).get("valid_ballots"), "errors": "; ".join(v.get("errors", [])), "warnings": "; ".join(v.get("warnings", []))}

## Run LLM stage

In [11]:
def run_llm_stage(pdf_paths, force=False):
    all_records = []
    reports = []
    failed = []
    manual_review = []
    stage_start = time.perf_counter()
    total = len(pdf_paths)

    for i, pdf in enumerate(pdf_paths):
        file_start = time.perf_counter()
        out_json = llm_json_path(str(pdf))
        txt_path = ocr_txt_path(str(pdf))
        form_label = detect_form_type(str(pdf))
        amphoe, tambon, unit_num = extract_path_metadata(str(pdf))

        if out_json.exists() and out_json.stat().st_size > 50 and not force:
            elapsed = time.perf_counter() - file_start
            print(f"[{i+1}/{total}] LLM SKIP: {pdf.name} | total={format_seconds(elapsed)}")
            try:
                result = json.loads(out_json.read_text(encoding="utf-8"))
                result["_validation"] = validate_record(result)
                all_records.extend(result_rows_from_json(result, str(pdf)))
                reports.append(validation_report_row(result, str(pdf)))
                if result["_validation"]["status"] == "REVIEW":
                    manual_review.append({"file": str(pdf), "ocr_text": str(txt_path), "json": str(out_json), "validation": result["_validation"]})
                LLM_FILE_TIMINGS.append({"file": str(pdf), "status": "SKIP", "rows": len(result.get("results", [])), "total_seconds": round(elapsed, 3)})
            except Exception as e:
                failed.append({"file": str(pdf), "stage": "load_existing_json", "error": str(e)})
                LLM_FILE_TIMINGS.append({"file": str(pdf), "status": "FAIL_LOAD_CACHE", "rows": 0, "total_seconds": round(elapsed, 3), "error": str(e)[:200]})
            continue

        print(f"[{i+1}/{total}] LLM: {amphoe}/{tambon}/หน่วย{unit_num} — {pdf.name}")
        try:
            if not txt_path.exists() or txt_path.stat().st_size <= 50:
                raise ValueError(f"Missing OCR text: {txt_path}")
            ocr_text = txt_path.read_text(encoding="utf-8")
            metadata = {"province": PROVINCE, "constituency": CONSTITUENCY, "amphoe": amphoe, "tambon": tambon, "unit": unit_num, "form_type": form_label, "source": str(pdf), "ocr_engine": OCR_MODEL, "llm_parser": LLM_MODEL}
            result = llm_parse_with_retry(ocr_text, metadata, max_retries=1)
            out_json.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
            all_records.extend(result_rows_from_json(result, str(pdf)))
            reports.append(validation_report_row(result, str(pdf)))
            if result["_validation"]["status"] == "REVIEW":
                manual_review.append({"file": str(pdf), "ocr_text": str(txt_path), "json": str(out_json), "validation": result["_validation"]})
            v = result["_validation"]
            elapsed = time.perf_counter() - file_start
            LLM_FILE_TIMINGS.append({"file": str(pdf), "status": v["status"], "rows": len(result.get("results", [])), "total_votes": v["total_votes_in_table"], "retry_count": v.get("retry_count", 0), "total_seconds": round(elapsed, 3)})
            print(f"  ✓ {v['status']} rows={len(result.get('results', []))} total_votes={v['total_votes_in_table']} total={format_seconds(elapsed)}")
        except Exception as e:
            elapsed = time.perf_counter() - file_start
            print(f"  ✗ LLM FAILED: {e} | total={format_seconds(elapsed)}")
            failed.append({"file": str(pdf), "stage": "llm", "error": str(e)})
            LLM_FILE_TIMINGS.append({"file": str(pdf), "status": "FAIL", "rows": 0, "total_seconds": round(elapsed, 3), "error": str(e)[:200]})

    if all_records:
        csv_path = Path(OUTPUT_DIR) / "all_results.csv"
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))
            writer.writeheader()
            writer.writerows(all_records)
        print(f"Master CSV → {csv_path} ({len(all_records)} rows)")

    if reports:
        report_path = Path(OUTPUT_DIR) / "validation_report.csv"
        with open(report_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=list(reports[0].keys()))
            writer.writeheader()
            writer.writerows(reports)
        print(f"Validation report → {report_path} ({len(reports)} files)")

    if failed:
        fail_path = Path(OUTPUT_DIR) / "_failed.json"
        fail_path.write_text(json.dumps(failed, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"Failed: {len(failed)} → {fail_path}")

    review_path = Path(OUTPUT_DIR) / "manual_review.json"
    review_path.write_text(json.dumps(manual_review, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Manual review → {review_path} ({len(manual_review)} files)")

    stage_elapsed = time.perf_counter() - stage_start
    print(f"LLM STAGE DONE: files={total}, failed={len(failed)}, review={len(manual_review)}, rows={len(all_records)}, total={format_seconds(stage_elapsed)}")
    write_timing_reports()
    return all_records, reports, failed, manual_review


## Run full pipeline

In [12]:
def get_all_pdfs():
    all_pdfs = sorted(Path(RAW_DATA_DIR).rglob("*.pdf"))
    print(f"Found: {len(all_pdfs)} PDFs")
    return all_pdfs

# Keep this helper if you still want one-command execution,
# but the recommended workflow is to run OCR and LLM in separate cells below.
def run_all(force_ocr=False, force_llm=False):
    all_pdfs = get_all_pdfs()
    ocr_failed = run_ocr_stage(all_pdfs, force=force_ocr)
    all_records, reports, failed, manual_review = run_llm_stage(all_pdfs, force=force_llm)
    print(f"Done. OCR failed={len(ocr_failed)}, LLM failed={len(failed)}, review={len(manual_review)}, rows={len(all_records)}")
    return ocr_failed, all_records, reports, failed, manual_review


## Execute OCR stage only


In [13]:
# Stage 1 only: OCR
# Run this first. It creates/updates files under ocr_texts/.
# Set force=True if you want to re-run OCR even when cached OCR text exists.

all_pdfs = get_all_pdfs()
ocr_failed = run_ocr_stage(all_pdfs, force=False)


Found: 566 PDFs
[1/566] OCR: rawData/อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วยเลือกตั้งที่ 1/5ทับ18(บช).pdf
    🖼️  PDF→image: 0.76s (4 pages)
    🔍 OCR page 1/4 (0.63 MB)... ✓ chars=2006 wait=0.00s api=6.59s total=6.59s
    🔍 OCR page 2/4 (0.59 MB)... ✓ chars=1588 wait=0.00s api=4.27s total=4.27s
    🔍 OCR page 3/4 (0.60 MB)... ✓ chars=1680 wait=0.00s api=6.02s total=6.02s
    🔍 OCR page 4/4 (0.43 MB)... ✓ chars=991 wait=0.00s api=4.39s total=4.39s
    💾 Saved → ocr_texts/อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วยเลือกตั้งที่ 1/5ทับ18(บช)_ocr.txt
  ✓ OCR FILE DONE: pages=4 chars=6331 total=22.49s
[2/566] OCR: rawData/อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วยเลือกตั้งที่ 1/5ทับ18.pdf
    🖼️  PDF→image: 0.32s (2 pages)
    🔍 OCR page 1/2 (0.58 MB)... ✓ chars=2430 wait=0.00s api=9.33s total=9.33s
    🔍 OCR page 2/2 (0.38 MB)... ✓ chars=1452 wait=0.00s api=4.69s total=4.69s
    💾 Saved → ocr_texts/อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วยเลือกตั้งที่ 1/5ทับ18_ocr.txt
  ✓ OCR FILE DONE: pages=2 chars=3904 total=14.54s
[3/566]

/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (98286371 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  ✗ OCR FAILED: Image size (211621888 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack. | total=26.67s
[374/566] OCR: rawData/เขต 2 อ.เมืองฯ/ตำบลขี้เหล็ก/ตำบลขี้เหล็ก/หน่วยเลือกตั้งที่ 1/5ทับ18.pdf


/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (100187373 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (100864946 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


    🖼️  PDF→image: 12.16s (2 pages)
    🔍 OCR page 1/2 (0.64 MB)... ✓ chars=2428 wait=0.00s api=6.86s total=6.86s
    🔍 OCR page 2/2 (0.47 MB)... ✓ chars=1049 wait=0.00s api=4.76s total=4.76s
    💾 Saved → ocr_texts/เขต 2 อ.เมืองฯ/ตำบลขี้เหล็ก/ตำบลขี้เหล็ก/หน่วยเลือกตั้งที่ 1/5ทับ18_ocr.txt
  ✓ OCR FILE DONE: pages=2 chars=3499 total=24.60s
[375/566] OCR: rawData/เขต 2 อ.เมืองฯ/ตำบลขี้เหล็ก/ตำบลขี้เหล็ก/หน่วยเลือกตั้งที่ 10/5ทับ18(บช).pdf


...

/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (103138250 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (92785819 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (101932250 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


    🖼️  PDF→image: 97.39s (4 pages)
    🔍 OCR page 1/4 (0.64 MB)... ✓ chars=2118 wait=0.00s api=5.48s total=5.48s
    🔍 OCR page 2/4 (0.55 MB)... ✓ chars=1547 wait=0.00s api=6.43s total=6.43s
    🔍 OCR page 3/4 (0.54 MB)... ✓ chars=1754 wait=0.00s api=4.25s total=4.25s
    🔍 OCR page 4/4 (0.54 MB)... ✓ chars=1056 wait=0.00s api=3.19s total=3.19s
    💾 Saved → ocr_texts/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่วยเลือกตั้งที่ 37/5ทับ18(บช)_ocr.txt
  ✓ OCR FILE DONE: pages=4 chars=6541 total=120.93s
[538/566] OCR: rawData/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่วยเลือกตั้งที่ 37/5ทับ18.pdf


/opt/miniconda3/envs/realsermsak/lib/python3.10/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (95198674 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


    🖼️  PDF→image: 41.73s (2 pages)
    🔍 OCR page 1/2 (0.64 MB)... ✓ chars=2475 wait=0.00s api=15.87s total=15.87s
    🔍 OCR page 2/2 (0.51 MB)... ✓ chars=990 wait=0.00s api=4.30s total=4.30s
    💾 Saved → ocr_texts/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่วยเลือกตั้งที่ 37/5ทับ18_ocr.txt
  ✓ OCR FILE DONE: pages=2 chars=3487 total=63.73s
[539/566] OCR: rawData/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่วยเลือกตั้งที่ 38/5ทับ18(บช).pdf
    🖼️  PDF→image: 71.90s (4 pages)
    🔍 OCR page 1/4 (0.52 MB)... ✓ chars=1658 wait=0.00s api=6.42s total=6.42s
    🔍 OCR page 2/4 (0.47 MB)... ✓ chars=1034 wait=0.00s api=3.11s total=3.11s
    🔍 OCR page 3/4 (0.64 MB)... ✓ chars=2438 wait=0.00s api=6.86s total=6.86s
    🔍 OCR page 4/4 (0.49 MB)... ✓ chars=944 wait=0.00s api=2.77s total=2.77s
    💾 Saved → ocr_texts/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่วยเลือกตั้งที่ 38/5ทับ18(บช)_ocr.txt
  ✓ OCR FILE DONE: pages=4 chars=6140 total=94.16s
[540/566] OCR: rawData/เขต 2 อ.เมืองฯ/เขตไร่น้อย/เขตไร่น้อย/หน่ว

ValueError: dict contains fields not in fieldnames: 'error'

In [16]:
def write_timing_reports():
    def _write_csv(path, rows):
        if not rows:
            return

        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        # collect every column used by any row
        fieldnames = sorted({k for row in rows for k in row.keys()})

        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
                extrasaction="ignore"
            )
            writer.writeheader()
            writer.writerows(rows)

        print(f"Timing report → {path} ({len(rows)} rows)")

    _write_csv(Path(OUTPUT_DIR) / "ocr_request_timings.csv", OCR_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "ocr_file_timings.csv", OCR_FILE_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "llm_request_timings.csv", LLM_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR) / "llm_file_timings.csv", LLM_FILE_TIMINGS)

In [17]:
write_timing_reports()

Timing report → output/ocr_request_timings.csv (1675 rows)
Timing report → output/ocr_file_timings.csv (566 rows)


## Execute LLM stage only


In [ ]:
# Stage 2 only: LLM extraction + validation
# Run this after OCR stage. It reads ocr_texts/ and writes JSON/CSV under output/.
# Set force=True if you want to re-run LLM parsing even when JSON already exists.

all_pdfs = get_all_pdfs()
all_records, reports, failed, manual_review = run_llm_stage(all_pdfs, force=False)
